###   __1__: Import necessary packages

In [1]:
include("include.jl");

###  __2__: Load the JLD2 file

In [2]:
data = MyOriginalPortfolioDataSet()

Dict{String, Any} with 1 entry:
  "dataset" => Dict{String, DataFrame}("DD"=>2329×8 DataFrame…

###  __3__: Extract the DataFrame from the loaded file

In [3]:
original_dataset = data["dataset"]

Dict{String, DataFrame} with 515 entries:
  "DD"   => 2329×8 DataFrame…
  "EMR"  => 2767×8 DataFrame…
  "CTAS" => 2767×8 DataFrame…
  "HSIC" => 2767×8 DataFrame…
  "KIM"  => 2767×8 DataFrame…
  "PLD"  => 2767×8 DataFrame…
  "IEX"  => 2767×8 DataFrame…
  "TPR"  => 1803×8 DataFrame…
  "BAC"  => 2767×8 DataFrame…
  "CBOE" => 2767×8 DataFrame…
  "EXR"  => 2767×8 DataFrame…
  "NCLH" => 2767×8 DataFrame…
  "CVS"  => 2767×8 DataFrame…
  "DRI"  => 2767×8 DataFrame…
  "DTE"  => 2767×8 DataFrame…
  "ZION" => 2767×8 DataFrame…
  "AVY"  => 2767×8 DataFrame…
  "EW"   => 2767×8 DataFrame…
  "EA"   => 2767×8 DataFrame…
  "NWSA" => 2767×8 DataFrame…
  "BBWI" => 859×8 DataFrame…
  "CAG"  => 2767×8 DataFrame…
  "GPC"  => 2767×8 DataFrame…
  "FCX"  => 2767×8 DataFrame…
  "GILD" => 2767×8 DataFrame…
  ⋮      => ⋮

In [4]:
haskey(original_dataset, "VXX")


true

In [9]:
vxx_data = original_dataset["VXX"]

Row,volume,volume_weighted_average_price,open,close,high,low,timestamp,number_of_transactions
,Float64,Float64,Float64,Float64,Float64,Float64,DateTime,Int64
1,18568.9,44208.5,44154.9,44195.8,44697.6,43612.2,2014-01-03T05:00:00,83739
2,25953.4,43607.1,43499.5,43755.5,44134.4,43151.4,2014-01-06T05:00:00,112824
3,19454.5,42860.9,43243.5,42721.3,43315.2,42557.4,2014-01-07T05:00:00,86711
4,15787.3,42826.6,42895.4,42772.5,43130.9,42496.0,2014-01-08T05:00:00,82134
5,16060.7,42798.1,42670.1,42833.9,43110.4,42496.0,2014-01-09T05:00:00,76119
6,23285.9,42288.7,42496.0,41820.2,43294.7,41738.2,2014-01-10T05:00:00,103796
7,40870.0,42374.0,41973.8,43325.4,44032.0,41236.5,2014-01-13T05:00:00,173228
8,46235.1,41833.8,42844.2,41594.9,42854.4,41513.0,2014-01-14T05:00:00,162696
9,19642.2,41734.2,41523.2,41789.4,42281.0,41390.1,2014-01-15T05:00:00,91661


###  __4__: Extract tickers with the maximum number of trading days as `AAPL`

Not all tickers in our dataset have the maximum number of trading days for various reasons, e.g., acquisition or de-listing events. Let's collect only those tickers with the maximum number of traditional days. First, let's compute the number of records for a company that we know has a maximum value, e.g., `AAPL,` and save that value in the `maximum_number_trading_days` variable:

In [5]:
maximum_number_trading_days = original_dataset["AAPL"] |> nrow

2767

Then, iterate through our data and collect only tickers with `maximum_number_trading_days` records. Save that data in the `dataset::Dict{String,DataFrame}` variable:

In [6]:
dataset = Dict{String,DataFrame}();
for (ticker,data) ∈ original_dataset
    if (nrow(data) == maximum_number_trading_days)
        dataset[ticker] = data;
    end
end

Lastly, let's get a sorted list of firms that we have in cleaned up `dataset` and save it in the `list_of_all_tickers::Array{String,1}` array

In [7]:
list_of_all_tickers = keys(dataset) |> collect |> x->sort(x)

424-element Vector{String}:
 "A"
 "AAL"
 "AAP"
 "AAPL"
 "ABBV"
 "ABT"
 "ACN"
 "ADBE"
 "ADI"
 "ADM"
 "ADP"
 "ADSK"
 "AEE"
 ⋮
 "WST"
 "WU"
 "WY"
 "WYNN"
 "XEL"
 "XOM"
 "XRAY"
 "XYL"
 "YUM"
 "ZBRA"
 "ZION"
 "ZTS"

### __5__: Split and save each ticker's data into two dictionaries

In [8]:
# train_start = DateTime(2014, 1, 1)
# train_end = DateTime(2023, 12, 31, 23, 59, 59)

# train_dataset = Dict{String, DataFrame}()
# test_dataset = Dict{String, DataFrame}()

# for ticker in list_of_all_tickers
#     df = dataset[ticker]
#     df.timestamp = DateTime.(df.timestamp)  # ensure the timestamp column is DateTime

#     train_data = filter(row -> train_start <= row.timestamp <= train_end, df)
#     test_data = filter(row -> row.timestamp > train_end, df)

#     train_dataset[ticker] = train_data
#     test_dataset[ticker] = test_data
# end

# train_file = joinpath(_PATH_TO_DATA, "train_dataset_2014_2023.jld2")
# test_file = joinpath(_PATH_TO_DATA, "test_dataset_2024_onward.jld2")

# @save train_file train_dataset
# @save test_file test_dataset